# Step 3 — Sentence Embeddings
Generates Sentence-BERT embeddings for all risk-labeled clauses and saves them for the Streamlit Q&A tab.

Outputs:
- `app/embeddings/clause_embeddings.npy`
- `app/embeddings/clause_ids.json`

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('../'))

import json
import numpy as np
import pandas as pd
from pathlib import Path

from src.semantic.embedder import generate_embeddings, save_embeddings

CLAUSES_PATH   = Path('../app/outputs/risk_reports/risk_labeled_clauses.csv')
EMBEDDINGS_DIR = Path('../app/embeddings')
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

EMB_PATH = EMBEDDINGS_DIR / 'clause_embeddings.npy'
IDS_PATH = EMBEDDINGS_DIR / 'clause_ids.json'

if not CLAUSES_PATH.exists():
    raise FileNotFoundError(f'Run step2_risk_scoring.ipynb first. Missing: {CLAUSES_PATH}')

df = pd.read_csv(CLAUSES_PATH)
print(f'Loaded {len(df)} clauses')
df.head()

Loaded 112 clauses


,clause_id,section,clause_text,source,document,word_count,clause_type,risk_score,risk_level,has_penalty,has_deadline,has_law_reference,explanation
0,data_protection_policy.txt_0000,7.1 Designation,REGULATORY DOCUMENT: DATA PROTECTION AND PRIVA...,data_protection_policy.txt,data_protection_policy.txt,34,condition,15,LOW,False,True,False,This is a LOW risk clause of general advisory ...
1,data_protection_policy.txt_0001,7.1 Designation,All entities subject to this regulation must c...,data_protection_policy.txt,data_protection_policy.txt,17,obligation,20,LOW,False,False,False,This is a LOW risk clause of general advisory ...
2,data_protection_policy.txt_0002,7.1 Designation,"""Personal data"" means any information relating...",data_protection_policy.txt,data_protection_policy.txt,13,general,0,LOW,False,False,False,This is a LOW risk clause of general advisory ...
3,data_protection_policy.txt_0003,7.1 Designation,"""Data controller"" refers to the entity that de...",data_protection_policy.txt,data_protection_policy.txt,16,general,0,LOW,False,False,False,This is a LOW risk clause of general advisory ...
4,data_protection_policy.txt_0004,7.1 Designation,"""Processing"" means any operation performed upo...",data_protection_policy.txt,data_protection_policy.txt,8,general,0,LOW,False,False,False,This is a LOW risk clause of general advisory ...


In [2]:
# Generate embeddings
texts = df['clause_text'].fillna('').tolist()
ids   = df['clause_id'].tolist()

print(f'Generating embeddings for {len(texts)} clauses...')
embeddings = generate_embeddings(texts, show_progress=True)
print(f'Embeddings shape: {embeddings.shape}')

Generating embeddings for 112 clauses...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embeddings shape: (112, 384)


In [3]:
# Save to disk
save_embeddings(embeddings, ids, str(EMB_PATH), str(IDS_PATH))
print(f'Embeddings saved to: {EMB_PATH}')
print(f'Clause IDs saved to: {IDS_PATH}')

# Verify
loaded = np.load(str(EMB_PATH))
print(f'\nVerification — loaded shape: {loaded.shape}')

with open(IDS_PATH) as f:
    loaded_ids = json.load(f)
print(f'Verification — loaded IDs: {len(loaded_ids)}')
print('\n✅ Step 3 complete. Launch the dashboard with: streamlit run app/app.py')

Embeddings saved to: ..\app\embeddings\clause_embeddings.npy
Clause IDs saved to: ..\app\embeddings\clause_ids.json

Verification — loaded shape: (112, 384)
Verification — loaded IDs: 112

✅ Step 3 complete. Launch the dashboard with: streamlit run app/app.py
